## סטטיסטיקה לכל ריצה

השיעור הזה הוא בעצם **השוואה**: אותה תוצאה בדיוק, בשתי דרכים שכבר מכירים — `groupby` (סעיף 10.5) מול צמצום לפי ציר על מערך NumPy (סעיף 9.7). ההשוואה עצמה היא הנקודה: מתי משתמשים בכל אחת, ולמה שתיהן חייבות לתת את אותו מספר.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna().reset_index(drop=True)

### דרך 1: groupby (pandas)

מתאימה כשהנתונים מגיעים כטבלה עם עמודת קיבוץ — בדיוק המצב שלנו, ישירות מ-`read_csv`.

In [ ]:
mean_via_groupby = df_clean.groupby("angle_deg")["range_measured"].mean()
print(mean_via_groupby)

### דרך 2: צמצום לפי ציר (NumPy)

דורשת שקודם נסדר את הנתונים כמערך `(זווית, ריצה)` — כלומר, שכל הזוויות יהיו עם **אותו מספר ריצות בדיוק**. זה בדיוק מקרה השימוש שממנו נבנה סעיף 9.7 (`Y.mean(axis=1)`), אבל כאן חייבים לוודא סדר וגודל אחיד לפני שממירים.

In [ ]:
counts_per_angle = df_clean.groupby("angle_deg").size()
print(counts_per_angle)   # לא אחיד! 7 בזוויות 15,45 (שם היה NaN שהוסר), 8 בשאר

# כדי לבנות מערך מלבני, לוקחים את אותו מספר מדידות (המינימלי המשותף) מכל קבוצה:
n_runs = int(counts_per_angle.min())
df_rect = df_clean.groupby("angle_deg").head(n_runs)
print(df_rect.groupby("angle_deg").size())   # עכשיו 7 בכל הזוויות

df_sorted = df_rect.sort_values(["angle_deg", "run_id"])
n_angles = df_sorted["angle_deg"].nunique()

Y = df_sorted["range_measured"].to_numpy().reshape(n_angles, n_runs)
mean_via_axis = Y.mean(axis=1)   # axis=1 (ריצה) נעלם - נשאר ציר הזווית

print(mean_via_axis)

### באג נפוץ: reshape על נתונים לא ממוינים

`reshape` (סעיף 9.3) לא יודע כלום על משמעות הנתונים — הוא רק קורא את הזיכרון ברצף. אם השורות מגיעות בסדר לא-ממוין (למשל, נתונים שמוזגו מכמה מפגשי מעבדה שונים ולא מוינו מחדש), ה-`reshape` **מערבב זוויות שונות** לתוך אותה "קבוצה" — בדיוק הבאג מסעיף 9.3, הפעם עם נתונים אמיתיים.

In [ ]:
df_shuffled = df_rect.sample(frac=1, random_state=7)   # מדמה נתונים שהגיעו בסדר לא ממוין

Y_wrong = df_shuffled["range_measured"].to_numpy().reshape(n_angles, n_runs)   # בלי מיון לפני!
print(np.allclose(Y_wrong.mean(axis=1), mean_via_axis))   # False - לא תואם, כי הסדר שונה

### נסו בעצמכם

חשבו את `mean_via_groupby` על `df_rect` (לא על `df_clean`, כדי שההשוואה תהיה הוגנת עם אותן 7 ריצות בכל זווית), והשוו ל-`mean_via_axis` עם `np.allclose`.

In [ ]:
# mean_via_groupby = df_rect.groupby("angle_deg")["range_measured"].mean()
# print(np.allclose(mean_via_groupby.to_numpy(), mean_via_axis))

`````{admonition} פתרון
:class: dropdown, tip
```python
mean_via_groupby = df_rect.groupby("angle_deg")["range_measured"].mean()
print(np.allclose(mean_via_groupby.to_numpy(), mean_via_axis))   # True
```
שתי הדרכים חייבות להתאים כשהן מחשבות על **אותן** נקודות בדיוק — שימו לב שהיינו צריכים לצמצם ל-7 ריצות בכל זווית (`df_rect`) כדי שההשוואה תהיה הוגנת; `mean_via_groupby` המקורי על `df_clean` (עם 7 או 8 ריצות, תלוי בזווית) לא היה שווה ל-`mean_via_axis`, לא כי אחת הדרכים שגויה, אלא כי הן פשוט חישבו על קבוצות נתונים שונות בגודלן.
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מתי עדיף להשתמש ב-groupby במקום בצמצום לפי ציר על מערך NumPy?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "תמיד - groupby תמיד עדיף", "correct": False, "feedback": "לא — כשהנתונים כבר מערך מסודר, axis reduction נוח וישיר יותר."},
            {"answer": "כשהנתונים כבר בטבלה עם עמודת קיבוץ, ולא בהכרח באותו מספר איברים לכל קבוצה", "correct": True, "feedback": "נכון."},
            {"answer": "groupby עובד רק על מספרים שלמים", "correct": False, "feedback": "לא נכון."},
            {"answer": "רק כשיש פחות מ-10 קבוצות", "correct": False, "feedback": "לא — אין מגבלה כזו; groupby עובד באותה יעילות עם כל מספר קבוצות."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

חשבו את סטיית התקן (לא הממוצע) של `v0_measured` בכל זווית, בשתי הדרכים (על `df_rect`, כדי לשמור על גודל קבוצה אחיד), וודאו שהן מתאימות עם `np.allclose`.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
std_via_groupby = df_rect.groupby("angle_deg")["v0_measured"].std().to_numpy()

df_sorted2 = df_rect.sort_values(["angle_deg", "run_id"])
Y_v0 = df_sorted2["v0_measured"].to_numpy().reshape(n_angles, n_runs)
std_via_axis = Y_v0.std(axis=1, ddof=1)

print(np.allclose(std_via_groupby, std_via_axis))
```
`````